**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# State-Space Models: Kalman → S4 → Mamba

The course only a signal processing society can teach properly. Modern sequence architectures (S4, Mamba) are *literally* this curriculum: state-space models ([Kalman](./Intro_AdFilt_KF.ipynb)), convolution kernels ([Foundations 1](../Intro_DSP/Foundations_of_Signal_Processing_1.ipynb)), and discretization ([Foundations 2](../Intro_DSP/Foundations_of_Signal_Processing_2.ipynb)) — rebranded for deep learning. Four sessions from the linear SSM you already know to a trained sequence model, with every identity verified numerically.

## 1. Pre-requisites

- [Kalman](./Intro_AdFilt_KF.ipynb) & [RNN](./Intro_RNN.ipynb) workshops.
- [Intro to Transformers](../Intro_DL_4_Physics/intro_transformers/intro_transformers.ipynb) for the architecture being challenged.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
torch.manual_seed(0); rng = np.random.default_rng(0)

---
### 🕐 Session 1 of 4 — *Linear SSMs Are Convolutions* (~35 min)
**Goal:** prove (numerically) that an LTI state space = one long FIR filter; why that unlocks parallel training.
**Builds on:** [Kalman](./Intro_AdFilt_KF.ipynb). &nbsp; **Feeds into:** Session 2 (HiPPO & discretization).

---

## 2. The Identity Everything Rests On

💡 **Intuition.** A linear time-invariant SSM $h_{t} = \bar{A} h_{t-1} + \bar{B} x_t, \; y_t = C h_t$ can be *unrolled*: $y_t = \sum_{k\ge0} C\bar{A}^{k}\bar{B} \, x_{t-k}$ — a *convolution* with kernel $K_k = C \bar A^k \bar B$. That one identity is the whole trick: **train as a convolution** (parallel, FFT-fast, no backprop-through-time vanishing) and **infer as a recurrence** (constant memory per step, unlike attention's growing KV cache). RNN pain and transformer pain, both dodged — for *linear* state dynamics.

In [ ]:
# ORACLE CHECK: recurrence output == convolution output, elementwise
# path 1: run the recurrence
# path 2: materialize the kernel, convolve

# YOUR CODE HERE


---
### 🕐 Session 2 of 4 — *HiPPO & Discretization: Why S4's A Matrix Is Special* (~35 min)
**Goal:** see why random A forgets; meet the memory-optimal initialization and the continuous-time view.
**Builds on:** Session 1; [Foundations 2](../Intro_DSP/Foundations_of_Signal_Processing_2.ipynb). &nbsp; **Feeds into:** Session 3 (training an SSM).

---

## 3. Long Memory Is an Initialization Problem

💡 **Intuition.** Session 1's kernel is a sum of geometric decays $\lambda_i^k$ ([Complex Analysis](../Intro_Math/Complex_Analysis/Complex_Analysis_Lite.ipynb): the poles are the modes!). Random stable $A$ ⇒ all modes die at similar rates ⇒ effective memory of a few dozen steps, the RNN disease in linear form. **HiPPO**'s insight: choose $A$ so the state stores *orthogonal-polynomial coefficients of the input's history* — a principled spread of timescales, kernels with long structured tails. S4 = HiPPO-initialized continuous SSM, discretized ([FoSP2 S2](../Intro_DSP/Foundations_of_Signal_Processing_2.ipynb)'s $\bar{A} = e^{A\Delta}$, in practice bilinear/ZOH) with a *learnable* step size $\Delta$ — the network literally learns its own sampling rate per channel.

In [ ]:
# Kernel shapes: random-diagonal vs HiPPO-style log-spaced timescales

# YOUR CODE HERE


---
### 🕐 Session 3 of 4 — *Train a Diagonal SSM* (~40 min)
**Goal:** build an S4-style layer (diagonal, conv-trained) and beat the LSTM on a long-memory task.
**Builds on:** Session 2. &nbsp; **Feeds into:** Session 4 (selectivity & Mamba).

---

## 4. The Layer, Assembled

Our layer (an honest simplification of S4D): per channel, learnable log-timescales $\lambda = e^{-e^{\theta}}$, learnable $C$; compute the kernel, convolve via FFT, add a skip and a nonlinearity. Trained **as a convolution**, verified equal to its recurrence.

In [ ]:
# ORACLE: layer's FFT-conv path == naive recurrence, for one channel

# YOUR CODE HERE


In [ ]:
# The long-memory gauntlet: recall the FIRST token's class after T=400 noise steps

# YOUR CODE HERE


---
### 🕐 Session 4 of 4 — *Selectivity & Mamba (Frontier Sketch)* (~30 min)
**Goal:** understand what Mamba changes — input-dependent dynamics — and what that costs.
**Builds on:** Session 3.

---

## 5. What Mamba Adds — and What It Breaks

> ℹ️ **Frontier sketch.** This session explains the mechanism and its trade-off; a full efficient selective-scan implementation (Mamba's hardware-aware kernel) is beyond a 40-minute session and is *not* implemented here.

💡 **Intuition.** Everything above is **LTI**: the same kernel for every input — the system cannot *decide* to remember this token and forget that one. Mamba makes $\bar B, \bar C, \Delta$ **functions of the current input** ('selective'): a content-controlled gate on the state, per step. The price is exactly the trade this course's structure predicts: input-dependent dynamics are **time-varying**, so the convolution identity of Session 1 *no longer holds* — no FFT training path. Mamba's contribution is showing the recurrence can still be computed fast on GPUs (parallel associative scan + kernel fusion — the [HW-Accelerated](../Intro_GPU/HW_Accelerated_Computing.ipynb) toolbox earning its keep).

The one-line summary of the whole architecture family:

| | RNN/LSTM | Transformer | S4 (LTI SSM) | Mamba (selective) |
|---|---|---|---|---|
| Train | sequential | parallel | parallel (FFT conv) | parallel (assoc. scan) |
| Infer/step | $O(1)$ | $O(T)$ (KV cache) | $O(1)$ | $O(1)$ |
| Content-dependent routing | gates | **attention** | ✗ | **selection** |
| DSP name | nonlinear IIR | data-adaptive kernel | FIR bank w/ learned poles | time-varying system |

In [ ]:
# The selectivity mechanism in miniature (correct but naive O(T) loop — the SKETCH):
# gate Δ_t = f(x_t) controls how much the state updates on each token
        # the gate sees the current token AND the previous one (a 1-step conv, as in Mamba's
        # local conv before the SSM) — Δ depends on the INPUT: this is the selectivity
# task LTI SSMs cannot do: output the last token that FOLLOWED a '2' marker

# YOUR CODE HERE


## 6. Conclusion

Linear SSM = convolution (verified), long memory = timescale spread (HiPPO's gift), training = FFT, inference = recurrence — and Mamba trades the conv identity for content-selective dynamics computed by scan. You can now read the S4/Mamba papers as *signal processing literature*, because that's what they are.

---
## Where next

- [LLMs from the Ground Up](../Intro_Mach_Learn/LLMs_from_the_Ground_Up.ipynb) — the model family SSMs compete with.
- [Kalman](./Intro_AdFilt_KF.ipynb) / [FoSP2](../Intro_DSP/Foundations_of_Signal_Processing_2.ipynb) — the two halves this course glued together.
- [Modern Architectures](../Intro_Mach_Learn/Modern_Architectures.ipynb) — where SSM blocks sit in today's model zoo.